# QuantJourney SDK - Corporate Actions and Adjustment Semantics

This notebook demonstrates a QuantJourney SDK workflow that checks adjusted price consistency, dividend history and corporate-action evidence around a single equity.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})

def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []

def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def max_drawdown(nav: pd.Series) -> float:
    drawdown = nav / nav.cummax() - 1
    return float(drawdown.min())

def performance_stats(ret: pd.Series) -> pd.Series:
    ret = ret.dropna()
    nav = (1 + ret).cumprod()
    ann_ret = nav.iloc[-1] ** (252 / len(ret)) - 1 if len(ret) and nav.iloc[-1] > 0 else np.nan
    ann_vol = ret.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) else np.nan
    return pd.Series({'annual_return': ann_ret, 'annual_volatility': ann_vol, 'sharpe': sharpe, 'max_drawdown': max_drawdown(nav) if len(nav) else np.nan, 'total_return': nav.iloc[-1] - 1 if len(nav) else np.nan})

def inverse_vol_weights(ret: pd.DataFrame) -> pd.Series:
    vol = ret.std().replace(0, np.nan)
    inv = 1 / vol
    return (inv / inv.sum()).fillna(0)

def min_variance_weights(ret: pd.DataFrame, ridge: float=0.0001) -> pd.Series:
    cov = ret.cov().fillna(0).to_numpy() * 252
    cov = cov + np.eye(cov.shape[0]) * ridge
    inv = np.linalg.pinv(cov)
    raw = inv @ np.ones(cov.shape[0])
    raw = np.maximum(raw, 0)
    if raw.sum() == 0:
        raw = np.ones(cov.shape[0])
    return pd.Series(raw / raw.sum(), index=ret.columns)

def portfolio_returns(ret: pd.DataFrame, weights: pd.Series) -> pd.Series:
    aligned = ret[weights.index].dropna()
    return aligned @ weights.reindex(aligned.columns).fillna(0)

def risk_contribution(ret: pd.DataFrame, weights: pd.Series) -> pd.Series:
    aligned = ret[weights.index].dropna()
    cov = aligned.cov() * 252
    w = weights.reindex(cov.columns).fillna(0).to_numpy()
    port_var = float(w @ cov.to_numpy() @ w)
    if port_var <= 0:
        return pd.Series(0.0, index=cov.columns)
    contrib = w * (cov.to_numpy() @ w) / port_var
    return pd.Series(contrib, index=cov.columns)

def rolling_betas(y: pd.Series, x: pd.DataFrame, window: int=126) -> pd.DataFrame:
    data = pd.concat([y.rename('asset'), x], axis=1).dropna()
    rows = []
    for i in range(window, len(data)):
        chunk = data.iloc[i - window:i]
        yy = chunk['asset'].to_numpy()
        xx = np.column_stack([np.ones(len(chunk)), chunk[x.columns].to_numpy()])
        beta = np.linalg.lstsq(xx, yy, rcond=None)[0][1:]
        rows.append(dict(date=data.index[i], **{col: beta[j] for j, col in enumerate(x.columns)}))
    return pd.DataFrame(rows).set_index('date') if rows else pd.DataFrame(columns=x.columns)

def zscore(s: pd.Series, window: int=252) -> pd.Series:
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()

def plot_nav(ret_map: dict[str, pd.Series], title: str) -> None:
    fig, ax = plt.subplots()
    for label, ret in ret_map.items():
        nav = (1 + ret.dropna()).cumprod()
        ax.plot(nav.index, nav, label=label)
    ax.set_title(title)
    ax.legend()
    plt.show()


In [ ]:
symbol = 'AAPL'
prices_raw = qj.eod.get_historical_prices(symbol=symbol, start_date='2018-01-01', end_date=END)
dividends_raw = qj.fmp.get_dividends_historical(symbol=symbol)
last_dividend_raw = qj.fmp.get_last_dividend(symbol=symbol)
shares_raw = qj.eod.get_shares_stats(symbol=symbol)
profile_raw = qj.fmp.get_company_profile(symbol=symbol)


In [ ]:
prices = pd.DataFrame(as_rows(prices_raw))
if not prices.empty:
    prices['date'] = pd.to_datetime(prices.get('date'), errors='coerce')
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in prices:
            prices[col] = pd.to_numeric(prices[col], errors='coerce')
    prices = prices.dropna(subset=['date']).sort_values('date').set_index('date')
    if 'adjusted_close' in prices and 'close' in prices:
        prices['adjustment_ratio'] = prices['adjusted_close'] / prices['close']
        prices['adjustment_gap'] = prices['adjusted_close'] - prices['close']
dividends = pd.DataFrame(as_rows(dividends_raw))
if not dividends.empty:
    date_col = next((col for col in dividends.columns if 'date' in str(col).lower()), dividends.columns[0])
    value_col = next((col for col in dividends.columns if 'dividend' in str(col).lower() or 'adj' in str(col).lower()), None)
    dividends['date'] = pd.to_datetime(dividends[date_col], errors='coerce')
    if value_col:
        dividends['dividend'] = pd.to_numeric(dividends[value_col], errors='coerce')
    dividends = dividends.dropna(subset=['date']).sort_values('date')


In [ ]:
audit = pd.Series({'price_rows': len(prices), 'dividend_rows': len(dividends), 'last_dividend_available': bool(as_rows(last_dividend_raw)), 'shares_stats_available': bool(as_rows(shares_raw)), 'profile_available': bool(as_rows(profile_raw)), 'has_adjusted_close': 'adjusted_close' in prices.columns if not prices.empty else False, 'adjustment_changes': int(prices['adjustment_ratio'].diff().abs().gt(0.001).sum()) if 'adjustment_ratio' in prices else 0})
display(audit)
if not prices.empty and {'close', 'adjusted_close'}.issubset(prices.columns):
    prices[['close', 'adjusted_close']].dropna().tail(1000).plot(title='Close vs adjusted close')
    plt.ylabel('price')
    plt.show()
if not dividends.empty and 'dividend' in dividends:
    dividends.tail(40).set_index('date')['dividend'].plot(kind='bar', title='Recent dividend events')
    plt.ylabel('cash dividend')
    plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.